# Building AI Agents with Tool Calling on Amazon Bedrock

This notebook shows how to build AI agents and enable LLMs to use external tools through the OpenAI-compatible **Amazon Bedrock `bedrock-mantle` endpoint**. It starts with manual text-based tool calls, progresses to JSON tool schemas and LangChain automation, and finishes with weather, web-search, MCP, and UI examples.

## What we compare

| Model | Bedrock model ID | Purpose in this notebook |
|---|---|---|
| Gemma 3 4B IT | `google.gemma-3-4b-it` | Compact model for prompt-based and native client-side tool-calling experiments |
| Ministral 3 8B | `mistral.ministral-3-8b-instruct` | Larger comparison model for native tool-calling behavior |

> `OpenAI` and `ChatOpenAI` are client interfaces here. The models are hosted by Amazon Bedrock, not by the OpenAI API.

## Learning Objectives  
* Understand why tool calling is useful and how LLMs can invoke external tools.
* Implement a minimal loop that parses the LLM's output and executes a Python function.
* See how *function schemas* (docstrings and type hints) let us scale to many tools.
* Connect to Amazon Bedrock through its OpenAI-compatible `bedrock-mantle` endpoint.
* Compare prompt-based tool calls with native client-side tool calling across models.
* Use **LangChain** to automate tool routing and execution.
* Combine LLM with a web‑search tool to build a simple ask‑the‑web agent.
* Connect to external tools using **MCP (Model Context Protocol)**, a universal standard for LLM‑tool integration.
* Optionally build a UI using Chainlit to test your agent.

## Roadmap
0. Environment setup
1. Write simple tools and connect them to an LLM
2. Standardize tool calling with JSON schemas
3. Use LangChain and LangGraph for automatic tool calling
4. Build a Perplexity-style web-search agent
5. (Optional) MCP: connect to external tool servers
6. (Optional) A minimal UI

# 0- Environment setup

### Step 1: Create your environment and install dependencies 
Before we start coding, you need a reproducible setup. Open a terminal in the same directory as this notebook, and use Conda or uv to install the project dependencies.

#### Option 1: Conda


```bash
# Create and activate the conda environment
conda env create -f environment.yaml && conda activate web_agent

```

#### Option 2: UV (faster)

If you prefer [uv](https://docs.astral.sh/uv/) over Conda:

```bash
# Install uv (skip if already installed)
curl -LsSf https://astral.sh/uv/install.sh | sh

# Create venv and install dependencies
uv venv .venv --python 3.11 && source .venv/bin/activate
uv pip install -r requirements.txt
```

### Step 2: Register this environment as a Jupyter kernel
```bash
python -m ipykernel install --user --name=web_agent --display-name "web_agent"
```
Now open your notebook and switch to the `web_agent` kernel (Kernel → Change Kernel).

### Step 3: Configure Amazon Bedrock

This notebook sends OpenAI-compatible Chat Completions requests to Amazon Bedrock. Create a Bedrock API key, keep it out of the notebook, and use the regional endpoint below.

```text
OPENAI_BASE_URL=https://bedrock-mantle.us-east-1.api.aws/v1
```

The next cells securely collect the API key and install the Python packages used by the examples. No local model server is required.

### Bedrock connection settings

The following code cell reads the Bedrock API key securely, sets the `bedrock-mantle` base URL, and configures the default project. Environment variables allow both the OpenAI SDK and LangChain's `ChatOpenAI` adapter to reuse the same connection settings.

In [1]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Amazon Bedrock API key: " )
os.environ.setdefault("OPENAI_BASE_URL", "https://bedrock-mantle.us-east-1.api.aws/v1")
os.environ.setdefault("OPENAI_PROJECT_ID", "default")
print(f"Configured Bedrock endpoint: {os.environ['OPENAI_BASE_URL']}")

Configured Bedrock endpoint: https://bedrock-mantle.us-east-1.api.aws/v1


### Install the agent dependencies

This cell installs the minimal LangChain packages used for tool definitions, OpenAI-compatible Bedrock models, and agent orchestration. Restart the kernel after installation if the imports are not immediately available.

In [2]:
%pip install langchain==1.2.15 langchain-core==1.2.26 langchain-openai==1.1.7

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1- Tool Calling

LLMs are strong at answering questions, but they cannot directly access external data such as live web results, APIs, or computations. In real applications, agents rarely rely only on their internal knowledge. They need to query APIs, retrieve data, or perform calculations to stay accurate and useful. Tool calling bridges this gap by allowing the LLM to request actions from the outside world.

<img src="assets/tools.png" width="700">

As show below, We first implement a tool, then describe the tool as part of the model's prompt. When the model decides that a tool is needed, it emits a structured output. A parser will detect this output, execute the corresponding function, and feed the result back to the LLM so the conversation continues.

<img src="assets/tool_flow.png" width="700">

In this section, you will implement `get_current_weather` and teach `google.gemma-3-4b-it`, accessed through Amazon Bedrock, when and how to request it.

### Build a real weather tool

This cell geocodes a city with Open-Meteo, retrieves its current conditions, and returns a concise string containing local time, temperature, latitude, and longitude. Returning text instead of printing it allows an agent to consume the result.

In [3]:
# ---------------------------------------------------------
# Step 1: Implement the tool
# ---------------------------------------------------------
# You can either:
#   (a) Call a real weather API (for example, OpenWeatherMap), or
#   (b) Create a dummy function that returns a fixed response (e.g., "It is 23°C and sunny in San Francisco.")
#
# Output:
#   • Return a short, human-readable sentence describing the weather.
#
# Example expected behavior:
#   get_current_weather("San Francisco") → "It is 23°C and sunny in San Francisco."
#

import json
from datetime import datetime
from urllib.error import URLError
from urllib.parse import urlencode
from urllib.request import urlopen

WEATHER_DESCRIPTIONS = {
    0: "clear skies", 1: "mainly clear skies", 2: "partly cloudy skies",
    3: "overcast skies", 45: "fog", 48: "freezing fog",
    51: "light drizzle", 53: "drizzle", 55: "heavy drizzle",
    56: "light freezing drizzle", 57: "freezing drizzle",
    61: "light rain", 63: "rain", 65: "heavy rain",
    66: "light freezing rain", 67: "freezing rain",
    71: "light snow", 73: "snow", 75: "heavy snow", 77: "snow grains",
    80: "light rain showers", 81: "rain showers", 82: "heavy rain showers",
    85: "light snow showers", 86: "heavy snow showers",
    95: "a thunderstorm", 96: "a thunderstorm with hail",
    99: "a severe thunderstorm with hail",
}

def _fetch_current_weather(city: str, unit: str) -> str:
    city = city.strip()
    unit = unit.strip().lower()
    if not city:
        return "Please provide a city name."
    if unit not in {"celsius", "fahrenheit"}:
        return "Unit must be either celsius or fahrenheit."

    try:
        geo_url = "https://geocoding-api.open-meteo.com/v1/search?" + urlencode(
            {"name": city, "count": 1, "language": "en", "format": "json"}
        )
        with urlopen(geo_url, timeout=10) as response:
            locations = json.load(response).get("results", [])

        if not locations:
            return f"I could not find a city named {city}."

        location = locations[0]
        weather_url = "https://api.open-meteo.com/v1/forecast?" + urlencode({
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "current": "temperature_2m,weather_code",
            "temperature_unit": unit,
            "timezone": "auto",
        })
        with urlopen(weather_url, timeout=10) as response:
            current = json.load(response)["current"]

        place = ", ".join(filter(None, [
            location.get("name"), location.get("admin1"), location.get("country")
        ]))
        description = WEATHER_DESCRIPTIONS.get(current["weather_code"], "unknown conditions")
        observed_at = datetime.fromisoformat(current["time"]).strftime("%B %d, %Y at %I:%M %p")
        temperature_symbol = "°C" if unit == "celsius" else "°F"
        return (
            f"As of {observed_at} local time, it is "
            f"{current['temperature_2m']:g}{temperature_symbol} with {description} in {place} "
            f"(latitude {location['latitude']:.4f}, longitude {location['longitude']:.4f})."
        )
    except (URLError, TimeoutError, json.JSONDecodeError, KeyError, TypeError, ValueError):
        return f"I could not retrieve the current weather for {city}."

def get_current_weather(city: str, unit: str = "celsius") -> str:
    """Return current weather, local time, and coordinates for a city.

    Args:
        city: City name to look up.
        unit: Temperature unit, either celsius or fahrenheit.
    """
    return _fetch_current_weather(city, unit)

### Describe the tool in a prompt

Before using native function calling, we teach the model a text protocol. For a weather question, the model should emit a `TOOL_CALL` JSON object instead of guessing the answer.

In [4]:
# ----------------------------------------------------------------------
# Step 2: Create a prompt to teach the LLM when and how to use your tool
# ----------------------------------------------------------------------
# What to include:
#   • A SYSTEM_PROMPT that tells the model about the tool use and describes the tool
#   • A USER_QUESTION with a user query that should trigger the tool.
#       Example: "What is the weather in San Diego today?"

SYSTEM_PROMPT = """You are a helpful assistant with access to this tool:
- get_current_weather(city: str, unit: str = \"celsius\"): Returns the current local date, time, temperature,
  weather conditions, latitude, and longitude for a city.

When the user asks about the current weather, do not guess or answer from memory.
Respond with exactly one tool request in this format and no additional text:
TOOL_CALL: {\"name\": \"get_current_weather\", \"args\": {\"city\": \"<city>\"}}
Use the city name supplied by the user in place of <city>.
For requests unrelated to current weather, answer normally."""

USER_QUESTION = "How is the weather in San Francisco today?"

### Send the prompt to Amazon Bedrock

This cell uses the OpenAI SDK against `bedrock-mantle`. Because this Gemma chat template expects user/assistant role alternation, the instructions and question are combined into one user message.

In [5]:
# ---------------------------------------------------------
# Step 3: Call the LLM with your prompt
# ---------------------------------------------------------
# Task:
#   Send SYSTEM_PROMPT + USER_QUESTION to the model.
#
# Steps:
#   1. Create an OpenAI-compatible client
#   2. Use chat.completions.create to send your prompt to google.gemma-3-4b-it
#   3. Print the response.
#
# Expected:
#   The model should return something like:
#   TOOL_CALL: {"name": "get_current_weather", "args": {"city": "San Diego"}}
# ---------------------------------------------------------
from openai import OpenAI

client = OpenAI()
prompt = f"{SYSTEM_PROMPT}\n\nUser question:\n{USER_QUESTION}"

response = client.chat.completions.create(
    model="google.gemma-3-4b-it",
    max_tokens=100,
    messages=[
        {"role": "user", "content": prompt},
    ],
)

model_output = response.choices[0].message.content
print(model_output)

TOOL_CALL: {"name": "get_current_weather", "args": {"city": "San Francisco"}}


### Parse and execute the requested tool

The model only requests an action; Python still executes it. This cell extracts the JSON payload, validates the tool name and arguments, calls the weather function, and prints its result.

In [6]:
# ---------------------------------------------------------
# Step 4: Manually parse the LLM output and call the tool
# ---------------------------------------------------------
# Task:
#   Detect when the model requests a tool, extract its name and arguments,
#   and execute the corresponding function.
#
# Steps:
#   1. Search for the text pattern "TOOL_CALL:{...}" in the model output.
#   2. Parse the JSON inside it to get the tool name and args.
#   3. Call the matching function (e.g., get_current_weather).
#
# Expected:
#   You should see a line like:
#       Calling tool `get_current_weather` with args {'city': 'San Diego'}
#       Result: It is 23°C and sunny in San Diego.
# ---------------------------------------------------------

import re, json, ast


def _extract_json_after_tool_call(text: str) -> str | None:
    m = re.search(r'TOOL_CALL\s*:\s*', text)
    if not m:
        return None
    start = m.end()
    idx = text.find('{', start)
    if idx == -1:
        return None
    i = idx
    stack = []
    while i < len(text):
        ch = text[i]
        if ch == '{':
            stack.append('{')
        elif ch == '}':
            if not stack:
                # unmatched closing brace
                return None
            stack.pop()
            if not stack:
                return text[idx:i + 1]
        i += 1
    return None

json_text = _extract_json_after_tool_call(model_output)
if not json_text:
    print("No TOOL_CALL JSON found or braces are unbalanced in model output")
else:
    try:
        try:
            payload = json.loads(json_text)
        except json.JSONDecodeError:
            # Try parsing Python-style dicts (single quotes) via ast.literal_eval
            payload = ast.literal_eval(json_text)
            if not isinstance(payload, dict):
                raise ValueError("Extracted payload is not a dict")

        tool_name = payload.get("name")
        tool_args = payload.get("args", {}) or {}

        print(f"Calling tool `{tool_name}` with args {tool_args}")

        tool_obj = globals().get(tool_name)
        if tool_obj is None:
            raise NameError(f"Tool '{tool_name}' not found in globals")

        # Support different tool object shapes
        if hasattr(tool_obj, "func") and callable(tool_obj.func):
            result = tool_obj.func(**tool_args)
        elif hasattr(tool_obj, "run") and callable(tool_obj.run):
            result = tool_obj.run(**tool_args)
        elif callable(tool_obj):
            result = tool_obj(**tool_args)
        else:
            raise TypeError(f"Tool '{tool_name}' is not callable")

        print("Result:", result)
    except Exception as e:
        print("Error parsing or calling tool:", repr(e))


Calling tool `get_current_weather` with args {'city': 'San Francisco'}
Result: As of July 31, 2026 at 10:00 PM local time, it is 14.9°C with mainly clear skies in San Francisco, California, United States (latitude 37.7749, longitude -122.4194).


# 2- Standardize tool calling

So far, we handled tool calling manually by writing a function, manually teaching the LLM about it, and write a regex to parse the output. This approach does not scale if we want to add more tools. Adding more tools would mean more `if/else` blocks and manual edits to the prompt.

To make the system flexible, we can standardize tool definitions by automatically reading each function's signature, converting it to a JSON schema, and passing that schema to the LLM. This way, the LLM can dynamically understand which tools exist and how to call them without requiring manual updates to prompts or conditional logic.

Next, you will implement a small helper that extracts metadata from functions and builds a schema for each tool.

### Generate a reusable tool schema

This cell converts a typed Python function into a JSON-compatible description. Models and agent frameworks use the schema to understand the tool name, purpose, argument types, defaults, and required fields.

In [7]:
# ---------------------------------------------------------
# Generate a JSON schema for a tool automatically
# ---------------------------------------------------------
#
# Steps:
#   1. Rewrite the get_current_weather function with docstring and arg types
#   2. Use `inspect.signature` to automatically get function parameters and docstring
#   2. For each argument, record its name, type, and description.
#   3. Build a schema containing: name, description, and parameters.
#   4. Test your helper on `get_current_weather` and print the result.
#
# Expected:
#   A dictionary describing the tool (its name, args, and types).
# ---------------------------------------------------------

from pprint import pprint
import inspect

def get_current_weather(city: str, unit: str = "celsius") -> str:
    """Return current weather, local time, and coordinates for a city.

    Args:
        city: City name to look up.
        unit: Temperature unit, either celsius or fahrenheit.
    """
    return _fetch_current_weather(city, unit)

def to_schema(fn):
    sig = inspect.signature(fn)
    doc = inspect.getdoc(fn) or ""
    type_map = {str: "string", int: "integer", float: "number", bool: "boolean"}
    descriptions = {}
    for line in doc.splitlines():
        name, separator, description = line.strip().partition(":")
        if separator and name in sig.parameters:
            descriptions[name] = description.strip()

    properties, required = {}, []
    for name, parameter in sig.parameters.items():
        parameter_schema = {
            "type": type_map.get(parameter.annotation, "string"),
            "description": descriptions.get(name, f"Value for {name}."),
        }
        if parameter.default is inspect.Parameter.empty:
            required.append(name)
        else:
            parameter_schema["default"] = parameter.default
        properties[name] = parameter_schema

    return {
        "name": fn.__name__,
        "description": doc.splitlines()[0] if doc else "",
        "parameters": {"type": "object", "properties": properties, "required": required},
    }


tool_schema = to_schema(get_current_weather)
pprint(tool_schema)

{'description': 'Return current weather, local time, and coordinates for a '
                'city.',
 'name': 'get_current_weather',
 'parameters': {'properties': {'city': {'description': 'City name to look up.',
                                        'type': 'string'},
                               'unit': {'default': 'celsius',
                                        'description': 'Temperature unit, '
                                                       'either celsius or '
                                                       'fahrenheit.',
                                        'type': 'string'}},
                'required': ['city'],
                'type': 'object'}}


### Give the model a tool menu

Instead of manually rewriting the prompt for every function, this cell serializes the generated schema and provides it to Gemma as a list of available tools. The response still uses the educational text-based `TOOL_CALL` format.

In [8]:
# ---------------------------------------------------------
# Provide the tool schema to the model instead of prompt surgery
# ---------------------------------------------------------
# Goal:
#   Give the model a "menu" of available tools so it can choose
#   which one to call based on the user’s question.
#
# Steps:
#   1. Add an extra system message (e.g., name="tool_spec")
#      containing the JSON schema(s) of your tools.
#   2. Include SYSTEM_PROMPT and the user question as before.
#   3. Send the messages to the model (google.gemma-3-4b-it).
#   4. Print the model output to see if it picks the right tool.
#
# Expected:
#   The model should produce a structured TOOL_CALL indicating
#   which tool to use and with what arguments.
# ---------------------------------------------------------


import json

tool_spec = json.dumps([tool_schema], indent=2)
prompt_with_tools = (
    f"{SYSTEM_PROMPT}\n\nAvailable tools:\n{tool_spec}"
    f"\n\nUser question:\n{USER_QUESTION}"
)

response = client.chat.completions.create(
    model="google.gemma-3-4b-it",
    max_tokens=150,
    messages=[
        {"role": "user", "content": prompt_with_tools},
    ],
)

model_output = response.choices[0].message.content
print(model_output)

TOOL_CALL: {"name": "get_current_weather", "args": {"city": "San Francisco"}}


## 3- LangChain for Tool Calling

So far, you built a simple tool-calling pipeline. While this exposes the mechanics, it does not scale well to multiple tools or multi-step workflows.

LangChain simplifies this process. Its agent loop passes tool schemas to the model, routes native tool calls to Python functions, returns observations to the model, and continues until a final answer is produced. This is a **ReAct-style** pattern: Reason, Act, Observe, and repeat.

<img src="assets/react.png" width="500">

The following links might be helpful for completing this section:
- [Create Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai)

### Register the weather function with LangChain

The `@tool` decorator turns the existing function into a LangChain tool. Its name, docstring, and type hints become the schema sent to a tool-capable model.

In [9]:
# ---------------------------------------------------------
# Step 1: Define tools for LangChain
# ---------------------------------------------------------
# Steps:
#   1. Keep your existing `get_current_weather` function as before.
#   2. Add the `@tool` decorator to your function so LangChain can register it automatically.
#
# Notes:
#   • The decorator converts your Python function into a standardized tool object.

from langchain_core.tools import tool

@tool
def get_current_weather(city: str, unit: str = "celsius") -> str:
    """Get current weather, local time, latitude, and longitude for a city."""
    return _fetch_current_weather(city, unit)


### Experiment 1: Gemma 3 4B IT

This cell creates a LangChain agent backed by `google.gemma-3-4b-it`. The test checks whether the model emits a native tool call for current weather instead of answering from its own knowledge.

In [10]:
# ---------------------------------------------------------
# Step 2: Create the Agent
# ---------------------------------------------------------
# Steps:
#   1. Create a google.gemma-3-4b-it LLM using the OpenAI-compatible API
#   2. Create the agent using create_agent
#   3. Test the agent with a natural question using agent.invoke

import os
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(
    model="google.gemma-3-4b-it",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"],
    max_tokens=200,
)

agent = create_agent(model=llm, tools=[get_current_weather])
result = agent.invoke({
    "messages": [{"role": "user", "content": "How is the weather in San Francisco today on 07/31/2026 at 4:00pm?"}]
})
print(result["messages"][-1].content)

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Okay, let's look at the weather forecast for San Francisco on July 31, 2026, at 4:00 PM.

Unfortunately, I cannot provide a *definitive* weather forecast for a date that is nearly 10 years in the future. Weather forecasting becomes increasingly unreliable the further out you go. 

However, I can give you a *likely* and *educated guess* based on historical weather patterns in San Francisco and trends in climate change:

**Likely Scenario (July 31, 2026, 4:00 PM):**

* **Temperature:** Expect a high around 68°F (20°C) and a low around 55°F (13°C).
* **Sky:** Partly cloudy. There will likely be some sunshine mixed with clouds.
* **Wind:** Moderate breezes from the west. Expect winds around 10-20


### Experiment 2: Ministral 3 8B

This cell repeats the agent test with `mistral.ministral-3-8b-instruct`. Keeping the tool and question similar makes each model's native tool-selection behavior easier to compare.

In [11]:
# ---------------------------------------------------------
# Step 2: Create the Agent
# ---------------------------------------------------------
# Steps:
#   1. Create a google.gemma-3-4b-it LLM using the OpenAI-compatible API
#   2. Create the agent using create_agent
#   3. Test the agent with a natural question using agent.invoke

import os
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(
    model="mistral.ministral-3-8b-instruct",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"],
    max_tokens=200,
)

agent = create_agent(model=llm, tools=[get_current_weather])
result = agent.invoke({
    "messages": [{"role": "user", "content": "How is the weather in San Francisco today?"}]
})
print(result["messages"][-1].content)

The current weather in San Francisco is **14.8°C (58.6°F)** with **mainly clear skies**. The local time is approximately **10:15 PM**.


### Compare the model results

Both Bedrock model cards list **client-side tool calling**, but the model still decides whether a tool is necessary when `tool_choice` is automatic. A model can therefore return ordinary text even when a tool schema was supplied.

| Model | Relative size | What to inspect |
|---|---:|---|
| `google.gemma-3-4b-it` | 4B | Did it emit `tool_calls`, or answer directly? |
| `mistral.ministral-3-8b-instruct` | 8B | Did it select the tool and produce valid arguments more consistently? |

Prompt-based `TOOL_CALL` text works without native function-calling support because your Python parser interprets ordinary text. LangChain agents instead expect a structured native tool-call object. Use the message trace—not just the final answer—to compare the models.

References: [Gemma 3 4B IT](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-google-gemma-3-4b-it.html) and [Ministral 3 8B](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-mistral-ai-ministral-3-8b.html).


### Retry the native tool-calling agent

This cell reruns the weather-agent workflow with Ministral. Compare its message trace and final response with the Gemma run above to see whether the tool was actually executed.

In [12]:
# ---------------------------------------------------------
# Step 2: Create the Agent
# ---------------------------------------------------------
# Steps:
#   1. Create a google.gemma-3-4b-it LLM using the OpenAI-compatible API
#   2. Create the agent using create_agent
#   3. Test the agent with a natural question using agent.invoke

import os
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(
    model="mistral.ministral-3-8b-instruct",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"],
    max_tokens=200,
)

agent = create_agent(model=llm, tools=[get_current_weather])
result = agent.invoke({
    "messages": [{"role": "user", "content": "How is the weather in San Francisco today?"}]
})
print(result["messages"][-1].content)

The weather in San Francisco today (July 31, 2026) at approximately 10:15 PM local time is:

- **Temperature**: 14.8°C (62.6°F)
- **Conditions**: Mainly clear skies

Would you like to know anything else about San Francisco? 😊


## 4- Web Search Agent

Now that you know how to use LangChain with tools, let's build something useful. Instead of a toy get_weather tool, let create an agent that searches the web and answers questions using real results. In the next section, you will create a [DuckDuckGo](https://github.com/deedy5/ddgs) search tool and wire it into a ReAct agent.

### Create the web-search tool

This cell wraps `DDGS().text()` with `@tool`. It returns titles, URLs, and snippets so an LLM can ground an answer in current web results rather than relying only on training data.

In [13]:
# ---------------------------------------------------------
# Step 1: Write a web search tool
# ---------------------------------------------------------
# Steps:
#   1. Write a function (e.g., search_web) that:
#        • Takes a query string
#        • Uses DuckDuckGo (DDGS) to fetch top results (titles + URLs)
#        • Returns them as a formatted string
#   2. Add the `@tool` decorator so LangChain can register it automatically.


from ddgs import DDGS
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for current information and return the top results."""
    results = DDGS().text(query, max_results=5)
    if not results:
        return f"No web results found for: {query}"
    return "\n\n".join(
        f"{index}. {result.get('title', 'Untitled')}\n"
        f"URL: {result.get('href', '')}\n"
        f"Snippet: {result.get('body', '')}"
        for index, result in enumerate(results, start=1)
    )


### Initialize the Ask-the-Web agent

This cell connects Gemma to the `search_web` tool through LangChain. `create_agent` manages the model/tool loop when the model returns a native search-tool request.

In [14]:
# ---------------------------------------------------------
# Step 2: Initialize the web-search agent
# ---------------------------------------------------------
# Steps:
#   1. Create an OpenAI-compatible LLM.
#   2. Add your `search_web` tool to the tools list.
#   3. Create the agent using create_agent.
#
# Expected:
#   The agent should be ready to accept user queries
#   and use your web search tool when needed.
# ---------------------------------------------------------

import os
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

web_llm = ChatOpenAI(
    model="google.gemma-3-4b-it",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"],
    max_tokens=500,
)

agent = create_agent(model=web_llm, tools=[search_web])

### Inspect an end-to-end search run

The test asks for time-sensitive information and prints every message. Look for an AI tool call, a tool-result message containing DDGS results, and a final sourced summary.

In [15]:
# ---------------------------------------------------------
# Step 3: Test your Ask-the-Web agent using agent.invoke
# ---------------------------------------------------------

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Use search_web to find the latest Python release and summarize the result with its source URL.",
    }]
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Use search_web to find the latest Python release and summarize the result with its source URL.
================================== Ai Message ==================================

```tool_code
print(search_web(queries=["latest python release"]))
```


## 5- (Optional) MCP: Model Context Protocol

Up to now, every tool you used started as a Python function you wrote and registered yourself. **MCP (Model Context Protocol)** lets you skip that step. Tools come from an external *server*, and your code just connects to it. Think of it like USB for AI tools: any MCP client can plug into any MCP server and immediately use whatever tools it offers.

Below, we connect to `mcp-server-fetch` (a ready-made server that can retrieve any URL) using the Python MCP SDK. We launch the server, discover its tools, and call one, all without writing a single `@tool` function. To learn more, read: https://github.com/modelcontextprotocol/servers/tree/main/src/fetch

> **LangChain integration:** The `langchain-mcp-adapters` package can convert MCP tools into LangChain-compatible tools automatically, so you can drop them straight into the agent like the ones in section 4.

### Connect to an MCP server

This optional cell launches an external MCP fetch server over standard input/output, opens a client session, discovers its tools, and demonstrates calling one without implementing it locally.

In [ ]:
# %pip install --upgrade --force-reinstall "mcp==1.26.0" "mcp-server-fetch==2026.7.10"

  Using cached mcp-1.26.0-py3-none-any.whl.metadata (89 kB)
  Using cached mcp_server_fetch-2026.7.10-py3-none-any.whl.metadata (8.5 kB)
  Using cached anyio-4.14.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached pydantic_settings-2.14.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pyjwt-2.13.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached pywin32-312-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached sse_starlette-3.4.6-py3-none-any.whl.metadata (15 kB)
  Using cached starlette-1.3.1-py3-none-any.whl.metadata (6.4 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-no

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
arxiv 2.4.1 requires requests~=2.32.0, but you have requests 2.34.2 which is incompatible.
cohere 6.1.0 requires pydantic-core<2.44.0,>=2.18.2, but you have pydantic-core 2.46.4 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Create an MCP client session and connect it to mcp-server-fetch.
# Follow this link: https://github.com/modelcontextprotocol/servers/tree/main/src/fetch

"""
YOUR CODE HERE (~10 lines of code)
"""
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command=sys.executable,
    args=["-m", "mcp_server_fetch"],
    env={"PYTHONIOENCODING": "utf-8"},
)

with open("mcp_fetch_stderr.log", "w", encoding="utf-8") as errlog:
    async with stdio_client(server_params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("Available tools:", [tool.name for tool in tools.tools])

            result = await session.call_tool(
                "fetch",
                arguments={"url": "https://www.python.org"},
            )
            print(result.content[0].text)

Available tools: ['fetch']
Contents of https://www.python.org/:
Welcome to Python.org

Notice: This page displays a fallback because interactive scripts did not run. Possible causes include disabled JavaScript or failure to load scripts or stylesheets.

Donate

≡ Menu

* A A

  + Smaller
  + Larger
  + Reset

* Socialize

  + LinkedIn
  + Mastodon
  + Chat on IRC
  + Twitter

* >\_ Launch Interactive Shell

* ```
  # Simple arithmetic >>> 1 / 2 0.5 >>> 2 ** 3 8 >>> 17 / 3 # true division returns a float 5.666666666666667 >>> 17 // 3 # floor division 5
  ```

  # Intuitive Interpretation

  Calculations are simple with Python, and expression syntax is straightforward: the operators +, -, \* and / work as expected; parentheses () can be used for grouping. More about simple math functions in Python 3.
* ```
  # Python 3: List comprehensions >>> fruits = ['Banana', 'Apple', 'Lime'] >>> loud_fruits = [fruit.upper() for fruit in fruits] >>> print(loud_fruits) ['BANANA', 'APPLE', 'LIME'] # Li

### Adapt MCP tools for LangChain

This cell converts discovered MCP tools into LangChain tools and attaches them to a Bedrock-backed agent. The agent can then select remote tools using the same interface as local Python functions.

In [ ]:
from langchain_mcp_adapters.tools import load_mcp_tools

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# Load the tool using load_mcp_tools
# create agent with llm and tools same as before
# Fetch the content of a website like http://python.org


"""
YOUR CODE HERE (~10 lines of code)
"""

## 6- (Optional) A Minimal UI

[Chainlit](https://chainlit.io/) is a Python library designed specifically for building LLM and agent UIs. It provides:
- Built-in streaming support
- Message history
- Step visualization (see tool calls as they happen)
- No frontend code required

If you are interested, follow Chainlit's documentation to implement a simple UI for your agent. The process typically involves:

1. You write a Python file named `chainlit_app.py` with the agent creation logic as well as UI handlers (e.g.,`@cl.on_message`)
2. Run the file in your terminal with `chainlit run app.py`
3. A web UI opens automatically at `http://localhost:8000`

### Generate a minimal Chainlit application

This optional cell writes a standalone UI application. The app recreates the Bedrock-backed search agent and exposes it through a browser chat interface with visible tool activity.

In [ ]:
%%writefile chainlit_app.py
# ---------------------------------------------------------
# Chainlit Web Search Agent
# ---------------------------------------------------------

import chainlit as cl
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from ddgs import DDGS

"""
YOUR CODE HERE
"""

# Step 1: Create a Bedrock LLM using ChatOpenAI
# Step 2: Write a search_web tool
# Step 3: Write a system prompt
# Step 4: Create your agent
# Step 5: Implement the Chainlit UI

## 🎉 Congratulations!
You have:

- Written simple tools and connected them to an LLM manually
- Learned how to generate JSON schemas for tools automatically
- Used LangChain to build a ReAct-style agent with tool calling
- Built a web search agent using DuckDuckGo
- Explored MCP (Model Context Protocol) for plug-and-play tools
- Deployed an interactive agent UI with Chainlit

👏 Great job! The techniques you implemented here power many production agents and chatbots.